In [ ]:
# !pip install ultralytics

In [ ]:
# !pip install ultralytics torchinfo sahi

In [ ]:
import numpy as np
from dataclasses import dataclass
import os
import math
import shutil
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary as torch_summary
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import wandb
import time
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import albumentations as A
import tqdm
# import fiftyone as fo

%matplotlib inline

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# wandb.login()

In [ ]:
@dataclass
class Config:
    patches_yolo_config_yaml: str
    original_yolo_config_yaml: str
    patches_train_images_folder: str
    patches_train_labels_folder: str
    patches_val_images_folder: str
    patches_val_labels_folder: str
    original_val_images_folder: str
    original_val_labels_folder: str
    training_output_folder: str
    saved_weights_filepath: str
    video_input_filepath: str
    video_output_filepath: str
    device: str

    # noinspection PyAttributeOutsideInit
    def init(self, training):
        self.training = training
        if self.training:
            os.makedirs(self.training_output_folder, exist_ok=True)

        self.class_labels = ['green', 'off', 'red', 'wait_on', 'yellow']

        self.seed = 8675309
        self.batch_size = 64
        self.starting_learning_rate = 3e-4
        self.max_epochs = 200
        self.patience = 50
        self.num_workers = 8 if self.device == 'cuda' else 0
        self.pin_memory = self.num_workers > 0
        self.image_size = 640
        self.original_image_width = 1920
        self.original_image_height = 1080
        self.use_amp = self.device == 'cuda'
        self.verbose = False

        self.patches_val_ids = np.array([Path(f).stem for f in os.listdir(config.patches_val_images_folder)])
        self.original_val_ids = np.array([Path(f).stem for f in os.listdir(config.original_val_images_folder)])

        self.model_name = 'yolo26x.pt'

        self.train_transforms = [
            # Lighting/Exposure (safe for color semantics)
            A.RandomBrightnessContrast(brightness_limit=(-0.2, 0.2), contrast_limit=(-0.2, 0.2), p=0.3),
            A.CLAHE(clip_limit=(1, 4), tile_grid_size=(8, 8), p=0.2),
            A.RandomGamma(gamma_limit=(80, 120), p=0.2),
            A.PlanckianJitter(mode='blackbody', p=0.15),

            # Blur/Noise (small kernels to preserve small-object detail)
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
            A.MotionBlur(blur_limit=(3, 5), p=0.1),
            A.GaussNoise(std_range=(0.03, 0.12), p=0.15),

            # Weather simulation
            A.RandomFog(fog_coef_range=(0.1, 0.3), alpha_coef=0.08, p=0.1),
            A.RandomRain(brightness_coefficient=0.8, rain_type='drizzle', p=0.08),
            A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 3),
                        shadow_intensity_range=(0.3, 0.5), p=0.1),

            # Image quality degradation
            A.ImageCompression(quality_range=(60, 95), p=0.15),
            A.Downscale(scale_range=(0.5, 0.9), p=0.1),
            A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.05, 0.2), p=0.1),

            # Sharpening (counterbalance blur augmentations)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.1),
        ]

        self.hsv_h=0.005    # reduced from 0.015 (color is critical)
        self.hsv_s=0.3      # reduced from 0.7 (avoid desaturating traffic lights)
        self.hsv_v=0.25     # reduced from 0.4 (albumentations adds brightness augs)

        self.sahi_slice_width = 640
        self.sahi_slice_height = 640
        sahi_horizontal_slices = 4
        sahi_vertical_slices = 2
        # use 4 horizontal slices and 2 vertical slices to match training
        # this code calculates the correct ratios
        self.sahi_overlap_width_ratio = 1 - (round((self.original_image_width - self.image_size) / (sahi_horizontal_slices - 1)) / self.image_size)
        self.sahi_overlap_height_ratio = 1 - (round((self.original_image_height - self.image_size) / (sahi_vertical_slices - 1)) / self.image_size)
        self.sahi_confidence_threshold = 0.25
        self.sahi_postprocess_type = "GREEDYNMM"
        self.sahi_postprocess_match_metric = "IOS"
        self.sahi_postprocess_match_threshold = 0.5
        self.sahi_postprocess_class_agnostic = False


config: Config = None
""" Set to environment-relevant config before training/inference """;

In [ ]:
local_config = Config(
    patches_yolo_config_yaml='data_patches_filtered/data.yaml',
    original_yolo_config_yaml='data/data.yaml',
    patches_train_images_folder='data_patches_filtered/train/images/',
    patches_train_labels_folder='data_patches_filtered/train/labels/',
    patches_val_images_folder='data_patches_filtered/val/images/',
    patches_val_labels_folder='data_patches_filtered/val/labels/',
    original_val_images_folder='data/val/images/',
    original_val_labels_folder='data/val/labels/',
    training_output_folder='data_gen/',
    saved_weights_filepath='data_gen/best.pt',
    video_input_filepath='data/inference_traffic_light_video.mp4',
    video_output_filepath='data_gen/traffic_light_detection_output_video.mp4',
    device='cpu',
)
kaggle_config = Config(
    patches_yolo_config_yaml='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/data.yaml',
    original_yolo_config_yaml='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/data.yaml',
    patches_train_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/train/images/',
    patches_train_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/train/labels/',
    patches_val_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/val/images/',
    patches_val_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/val/labels/',
    original_val_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/val/images/',
    original_val_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/val/labels/',
    training_output_folder='/kaggle/working/',
    saved_weights_filepath='/kaggle/input/datasets/kyledunne/traffic-lights-yolo-best-weights/best.pt',
    video_input_filepath='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/inference_traffic_light_video.mp4',
    video_output_filepath='/kaggle/working/traffic_light_detection_output_video.mp4',
    device='cuda',
)
colab_home_folder = '/content/drive/My Drive/Colab Notebooks/traffic-light-detection/'
colab_config = Config(
    patches_yolo_config_yaml=colab_home_folder + 'data_patches_filtered/data.yaml',
    original_yolo_config_yaml=colab_home_folder + 'data/data.yaml',
    patches_train_images_folder=colab_home_folder + 'data_patches_filtered/train/images/',
    patches_train_labels_folder=colab_home_folder + 'data_patches_filtered/train/labels/',
    patches_val_images_folder=colab_home_folder + 'data_patches_filtered/val/images/',
    patches_val_labels_folder=colab_home_folder + 'data_patches_filtered/val/labels/',
    original_val_images_folder=colab_home_folder + 'data/val/images/',
    original_val_labels_folder=colab_home_folder + 'data/val/labels/',
    training_output_folder=colab_home_folder + 'data_gen/',
    saved_weights_filepath=colab_home_folder + 'data_gen/best.pt',
    video_input_filepath=colab_home_folder + 'data/inference_traffic_light_video.mp4',
    video_output_filepath=colab_home_folder + 'data_gen/traffic_light_detection_output_video.mp4',
    device='cuda',
)

In [ ]:
def plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=None,
        pred_bounding_boxes_class_labels=None,
        pred_boxes_confidence=None,
        true_bounding_boxes=None,
        true_bounding_boxes_class_labels=None,
        white_text_background=True,
):
    for i, image in enumerate(images):
        h, w = image.shape[:2]
        fig, ax = plt.subplots(1)
        ax.imshow(image)
        ax.axis('off')

        if true_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(true_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='orange', facecolor='none')
                ax.add_patch(rect)
                if true_bounding_boxes_class_labels is not None:
                    label = true_bounding_boxes_class_labels[i][j]
                    if white_text_background:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left')

        if pred_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(pred_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                if pred_boxes_confidence is not None:
                    conf_str = f"{int(pred_boxes_confidence[i][j] * 100)}%"
                    if pred_bounding_boxes_class_labels is not None:
                        label = f"{pred_bounding_boxes_class_labels[i][j]} {conf_str}"
                    else:
                        label = conf_str
                    if white_text_background:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left')

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_val_images_with_ground_truth_bounding_boxes(num_images_to_show=3, white_text_background=True):
    ids = np.random.choice(config.original_val_ids, num_images_to_show)
    images = []
    all_boxes = []
    all_class_labels = []
    for image_id in ids:
        image_path = f'{config.original_val_images_folder}{image_id}.jpg'
        image_label = f'{config.original_val_labels_folder}{image_id}.txt'
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        images.append(image)
        boxes = []
        box_labels = []
        with open(image_label, 'r') as label_file:
            for line in label_file:
                line_segments = line.strip().split()
                box_labels.append(config.class_labels[int(line_segments[0])])
                coords = [float(c) for c in line_segments[-4:]]
                boxes.append(coords)
        all_boxes.append(boxes)
        all_class_labels.append(box_labels)
    plot_images_with_bounding_boxes(images, true_bounding_boxes=all_boxes, true_bounding_boxes_class_labels=all_class_labels, white_text_background=white_text_background)

In [ ]:
def train_yolo():
    start_time = time.time()
    print(f't=0: Initializing training setup...')

    # Initialize wandb run (logs hyperparams and enables automatic metric logging)
    wandb.init(
        project='traffic-light-detection',
        name=f'run={int(start_time)}',
        config={
            'model': config.model_name,
            'epochs': config.max_epochs,
            'batch_size': config.batch_size,
            'image_size': config.image_size,
            'lr0': config.starting_learning_rate,
            'patience': config.patience,
            'seed': config.seed,
            'amp': config.use_amp,
            'device': config.device,
        }
    )

    model = YOLO(config.model_name)

    # --- wandb metric logging via Ultralytics callbacks ---
    def on_train_epoch_end(trainer):
        """Log training losses and learning rates after each training epoch."""
        epoch = trainer.epoch + 1
        loss_items = trainer.label_loss_items(trainer.tloss, prefix='train')
        metrics = {}
        for key in ('train/box_loss', 'train/cls_loss', 'train/dfl_loss'):
            if key in loss_items:
                metrics[key] = loss_items[key]
        for key in ('lr/pg0', 'lr/pg1', 'lr/pg2'):
            if key in trainer.lr:
                metrics[key] = trainer.lr[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    def on_fit_epoch_end(trainer):
        """Log validation metrics after each validation pass."""
        epoch = trainer.epoch + 1
        metrics = {}
        for key in ('metrics/precision(B)', 'metrics/recall(B)',
                     'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
                     'val/box_loss', 'val/cls_loss', 'val/dfl_loss'):
            if key in trainer.metrics:
                metrics[key] = trainer.metrics[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    model.add_callback('on_train_epoch_end', on_train_epoch_end)
    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

    # Train — verbose=True enables per-epoch cell output logging
    results = model.train(
        data=config.patches_yolo_config_yaml,
        epochs=config.max_epochs,
        imgsz=config.image_size,
        batch=config.batch_size,
        optimizer='AdamW',
        lr0=config.starting_learning_rate,
        warmup_bias_lr=0.01,
        cos_lr=True,
        patience=config.patience,
        seed=config.seed,
        amp=config.use_amp,
        workers=config.num_workers,
        device=config.device,
        verbose=True,
        plots=True,
        lrf=0.2,
        augmentations=config.train_transforms,
        hsv_h=config.hsv_h,
        hsv_s=config.hsv_s,
        hsv_v=config.hsv_v,
    )

    # Copy best model weights to config.training_output_folder
    best_src = Path(results.save_dir) / 'weights' / 'best.pt'
    os.makedirs(config.training_output_folder, exist_ok=True)
    best_dst = Path(config.training_output_folder) / 'best.pt'
    shutil.copy(str(best_src), str(best_dst))
    print(f'Best model saved to: {best_dst}')

    # Upload best model to wandb as an artifact (replaces add_wandb_callback checkpointing,
    # which is broken in kaggle environment due to unresolved imports in the callback module)
    artifact = wandb.Artifact(name='best-model', type='model')
    artifact.add_file(str(best_dst))
    wandb.log_artifact(artifact)

    wandb.finish()
    return results

In [ ]:
def _build_coco_gt_and_eval(coco_preds, label):
    """
    Build COCO ground-truth from val label files and evaluate predictions.

    Parameters
    ----------
    coco_preds : list[dict]
        Predictions in COCO format (image_id, category_id, bbox, score).
        image_id must already be an integer matching stem_to_id mapping.
    label : str
        Section header for printed output (e.g. "COCO Metrics").
    """
    images_dir = Path(config.original_val_images_folder)
    labels_dir = Path(config.original_val_labels_folder)

    all_stems = sorted(p.stem for p in images_dir.glob('*.jpg'))
    stem_to_id = {stem: i + 1 for i, stem in enumerate(all_stems)}

    if not coco_preds:
        print(f'\n--- {label} (pycocotools) ---')
        print('  No predictions found — skipping COCOeval.')
        return stem_to_id

    # Build COCO ground-truth structure in memory
    coco_images = []
    coco_annotations = []
    ann_id = 1

    for stem in all_stems:
        img = cv2.imread(str(images_dir / f'{stem}.jpg'))
        h, w = img.shape[:2] if img is not None else (640, 640)
        int_id = stem_to_id[stem]

        coco_images.append({'id': int_id, 'file_name': f'{stem}.jpg', 'width': w, 'height': h})

        label_path = labels_dir / f'{stem}.txt'
        if not label_path.exists() or label_path.stat().st_size == 0:
            continue

        with open(label_path) as lf:
            for line in lf:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_idx, xc, yc, bw, bh = map(float, parts)
                x = (xc - bw / 2) * w
                y = (yc - bh / 2) * h
                bw_px, bh_px = bw * w, bh * h
                coco_annotations.append({
                    'id': ann_id, 'image_id': int_id, 'category_id': int(cls_idx),
                    'bbox': [x, y, bw_px, bh_px], 'area': bw_px * bh_px, 'iscrowd': 0,
                })
                ann_id += 1

    # Load into pycocotools without writing any files
    coco_gt = COCO()
    coco_gt.dataset = {
        'images': coco_images,
        'annotations': coco_annotations,
        'categories': [{'id': i, 'name': name, 'supercategory': 'object'} for i, name in enumerate(config.class_labels)],
    }
    coco_gt.createIndex()
    coco_dt = coco_gt.loadRes(coco_preds)

    # Run evaluation and print all 12 standard COCO metrics
    print(f'\n--- {label} (pycocotools) ---')
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return stem_to_id


def print_all_coco_metrics(val_results):
    """
    Run pycocotools COCOeval and print all 12 standard COCO metrics.

    Ultralytics only triggers pycocotools automatically for officially-structured
    COCO datasets (is_coco=True). This dataset uses hex-string filenames, so even
    with the COCO layout, image IDs in predictions.json would be strings while
    pycocotools requires integers — COCOeval would score zeros. Instead, we build
    the COCO ground-truth in memory and remap IDs consistently for both GT and
    predictions before running COCOeval.

    Requires model.val() to have been called with save_json=True so that
    predictions.json exists in val_results.save_dir.
    """
    pred_json_path = Path(val_results.save_dir) / 'predictions.json'
    images_dir = Path(config.original_val_images_folder)

    # Load predictions (image_id is a hex string, e.g. "b01d46f9911d558b")
    with open(pred_json_path, 'r') as f:
        raw_preds = json.load(f)

    # pycocotools requires integer IDs. Build a stable str→int mapping from
    # all val images (not just predicted ones, so recall denominator is correct).
    all_stems = sorted(p.stem for p in images_dir.glob('*.jpg'))
    stem_to_id = {stem: i + 1 for i, stem in enumerate(all_stems)}

    # Remap predictions to integer IDs
    remapped_preds = [
        {**pred, 'image_id': stem_to_id[pred['image_id']], 'category_id': pred['category_id'] - 1}
        for pred in raw_preds
        if pred['image_id'] in stem_to_id
    ]

    _build_coco_gt_and_eval(remapped_preds, "COCO Metrics")


def print_all_coco_metrics_sahi(sahi_predictions):
    """
    Compute and print all 12 standard COCO metrics from SAHI predictions.

    This allows direct comparison with baseline model.val() metrics. Uses the same
    ground-truth building logic as print_all_coco_metrics(), but reads predictions
    from SAHI result objects instead of predictions.json.
    """
    images_dir = Path(config.original_val_images_folder)

    all_stems = sorted(p.stem for p in images_dir.glob('*.jpg'))
    stem_to_id = {stem: i + 1 for i, stem in enumerate(all_stems)}

    # Build predictions in COCO format from SAHI results
    coco_preds = []
    for image_path, sahi_result in sahi_predictions:
        stem = Path(image_path).stem
        if stem not in stem_to_id:
            continue
        int_id = stem_to_id[stem]
        img = cv2.imread(image_path)
        h, w = img.shape[:2] if img is not None else (640, 640)

        for pred in sahi_result.object_prediction_list:
            bbox = pred.bbox
            x1, y1, x2, y2 = bbox.minx, bbox.miny, bbox.maxx, bbox.maxy
            bw = x2 - x1
            bh = y2 - y1
            cat_id = config.class_labels.index(pred.category.name)
            coco_preds.append({
                'image_id': int_id,
                'category_id': cat_id,
                'bbox': [x1, y1, bw, bh],
                'score': pred.score.value,
            })

    _build_coco_gt_and_eval(coco_preds, "SAHI COCO Metrics")

In [ ]:
def create_sahi_detection_model():
    return AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=config.saved_weights_filepath,
        confidence_threshold=config.sahi_confidence_threshold,
        device=config.device,
        image_size=config.image_size,
    )


def run_sahi_on_image(detection_model, image):
    """Run SAHI sliced prediction on a single image (file path or numpy array)."""
    return get_sliced_prediction(
        image=image,
        detection_model=detection_model,
        slice_height=config.sahi_slice_height,
        slice_width=config.sahi_slice_width,
        overlap_height_ratio=config.sahi_overlap_height_ratio,
        overlap_width_ratio=config.sahi_overlap_width_ratio,
        postprocess_type=config.sahi_postprocess_type,
        postprocess_match_metric=config.sahi_postprocess_match_metric,
        postprocess_match_threshold=config.sahi_postprocess_match_threshold,
        postprocess_class_agnostic=config.sahi_postprocess_class_agnostic,
        perform_standard_pred=False,
        verbose=0,
    )


def sahi_result_to_viz_format(sahi_result, image):
    """Convert SAHI PredictionResult to normalized xywh center-format for plot_images_with_bounding_boxes()."""
    h, w = image.shape[:2]
    boxes = []
    confidences = []
    labels = []
    for pred in sahi_result.object_prediction_list:
        bbox = pred.bbox
        x1, y1, x2, y2 = bbox.minx, bbox.miny, bbox.maxx, bbox.maxy
        bw = x2 - x1
        bh = y2 - y1
        x_center = (x1 + bw / 2) / w
        y_center = (y1 + bh / 2) / h
        boxes.append([x_center, y_center, bw / w, bh / h])
        confidences.append(pred.score.value)
        labels.append(pred.category.name)
    return boxes, confidences, labels

In [ ]:
def validate_yolo_original_setup():
    model = YOLO(config.saved_weights_filepath)
    val_results = model.val(
        data=config.patches_yolo_config_yaml,
        imgsz=config.image_size,
        batch=config.batch_size,
        workers=config.num_workers,
        device=config.device,
        verbose=True,
        plots=True,
        save_json=True,
    )
    print_all_coco_metrics(val_results)

In [ ]:
def predict_yolo(return_preds=True, print_coco_metrics=True):
    model = YOLO(config.saved_weights_filepath)

    # Run validation — logs all available COCO metrics to stdout.
    # save_json=True saves predictions.json to val_results.save_dir, which is read
    # by print_all_coco_metrics() to compute all 12 standard COCO metrics.
    # verbose=True prints the per-class mAP table from ultralytics.
    val_results = model.val(
        data=config.original_yolo_config_yaml,
        imgsz=config.image_size,
        batch=config.batch_size,
        workers=config.num_workers,
        device=config.device,
        verbose=True,
        plots=True,
        save_json=True,
    )

    if print_coco_metrics:
        print_all_coco_metrics(val_results)

    if not return_preds:
        return

    # Run SAHI sliced inference on each val image to collect detections at native
    # resolution (matching the 640x640 training patch size), instead of model.predict()
    # which resizes full images down to 640px and loses small objects.
    detection_model = create_sahi_detection_model()
    image_paths = sorted(Path(config.original_val_images_folder).glob('*.jpg'))
    predictions = []
    for image_path in tqdm.tqdm(image_paths, desc='SAHI Inference on validation set images'):
        sahi_result = run_sahi_on_image(detection_model, str(image_path))
        predictions.append((str(image_path), sahi_result))

    return predictions

In [ ]:
def predict_yolo_with_visualizations(num_to_visualize=3, print_coco_metrics=False):
    predictions = predict_yolo(print_coco_metrics=print_coco_metrics)

    indices = np.random.choice(len(predictions), num_to_visualize, replace=False)
    subset = [predictions[i] for i in indices]

    images = []
    pred_bounding_boxes = []
    pred_boxes_confidence = []
    pred_class_labels = []
    for image_path, sahi_result in subset:
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        images.append(image)
        boxes, confidences, labels = sahi_result_to_viz_format(sahi_result, image)
        pred_bounding_boxes.append(boxes)
        pred_boxes_confidence.append(confidences)
        pred_class_labels.append(labels)

    plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=pred_bounding_boxes,
        pred_bounding_boxes_class_labels=pred_class_labels,
        pred_boxes_confidence=pred_boxes_confidence,
    )

In [ ]:
def predict_video():
    detection_model = create_sahi_detection_model()

    video = cv2.VideoCapture(config.video_input_filepath)
    if not video.isOpened():
        print('Error opening video file')
        return

    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)

    out_dir = os.path.dirname(config.video_output_filepath)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    output_file = cv2.VideoWriter(
        filename=config.video_output_filepath,
        fourcc=cv2.VideoWriter_fourcc(*'mp4v'),
        fps=float(fps),
        frameSize=(width, height),
        isColor=True,
    )

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        # Convert BGR to RGB for SAHI (its ultralytics adapter internally handles RGB-to-BGR)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        sahi_result = run_sahi_on_image(detection_model, frame_rgb)

        for pred in sahi_result.object_prediction_list:
            bbox = pred.bbox
            x1, y1, x2, y2 = int(bbox.minx), int(bbox.miny), int(bbox.maxx), int(bbox.maxy)
            conf = pred.score.value
            label = f'{pred.category.name} {int(conf * 100)}%'

            cv2.rectangle(frame, (x1, y1), (x2, y2), color=(0, 165, 255), thickness=2)
            (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - th - baseline - 5), (x1 + tw, y1), color=(255, 255, 255), thickness=-1)
            cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                        fontScale=0.6, color=(0, 165, 255), thickness=2)

        output_file.write(frame)

    video.release()
    output_file.release()
    print(f'Video saved to: {config.video_output_filepath}')

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret('wandb_key')
# !wandb login $wandb_key

In [ ]:
config = local_config
config.init(training=False)
validate_yolo_original_setup()